# Docker

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/docker/docker.ipynb)

Docker packages a job with its exact runtime (OS libraries, Python, JVM, drivers) into an image that runs the same on a laptop, in CI and on Kubernetes. For data engineers that means reproducible Spark and Airflow images, local stacks with Compose, and the build, security and resource habits that keep pipelines fast and stable. Dockerfiles and Compose files appear as snippets; the build cache, `.dockerignore`, Compose startup ordering and memory sizing are simulated in Python, so nothing here calls `docker`.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"yaml": "pyyaml>=6.0"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Images, layers and containers

+ An **image** is an immutable, ordered list of layers plus a JSON config (entrypoint, env, user, working directory, exposed ports), addressed by a content digest (`sha256:...`).
+ A **layer** is the filesystem diff produced by one build instruction (`RUN`, `COPY`, `ADD`). Layers are shared between images that have the same parents, so ten images on `python:3.12-slim` store the base once.
+ A **container** is a running image plus a thin writable layer, isolated with Linux namespaces (process, network, mount) and limited with cgroups (CPU, memory). The writable layer disappears with the container, so data that must survive goes in a volume.
+ A **registry** (Docker Hub, ECR, GCR/Artifact Registry, GHCR) stores images; a **tag** is a mutable pointer to a digest.

Containers share the host kernel, which is why they start in milliseconds and why they are a weaker isolation boundary than virtual machines.

## The build cache

Each instruction's cache key is derived from the parent layer, the instruction text and, for `COPY`/`ADD`, the checksums of the copied files. The first miss invalidates every later step. So order instructions from least to most frequently changing: base image, system packages, dependency lockfile and install, then source code.

### Where you'll use it

- **Reproducible jobs:** the same image runs a Spark or dbt job on a laptop, in CI and in production, so "works on my machine" stops being a reason for a failed run.

In [ ]:
STEPS = [
    ("FROM python:3.12-slim", None),
    ("COPY requirements.txt .", "requirements.txt"),
    ("RUN pip install -r requirements.txt", None),
    ("COPY src/ ./src/", "src/"),
    ("RUN python -m compileall src", None),
]


def build(changed: set[str], steps: list[tuple[str, str | None]] = STEPS) -> list[str]:
    outcomes, cache_broken = [], False
    for _instruction, copied in steps:
        cache_broken = cache_broken or (copied in changed)
        outcomes.append("rebuild" if cache_broken else "cached")
    return outcomes


assert build({"requirements.txt"}) == ["cached", "rebuild", "rebuild", "rebuild", "rebuild"]
assert build({"src/"}) == ["cached", "cached", "cached", "rebuild", "rebuild"]

# Anti-pattern: copying everything before installing reruns pip on every code edit.
COPY_ALL_FIRST = [("FROM python:3.12-slim", None), ("COPY . .", "src/"), ("RUN pip install -r requirements.txt", None)]
assert build({"src/"}, COPY_ALL_FIRST) == ["cached", "rebuild", "rebuild"]

for changed in [{"requirements.txt"}, {"src/"}]:
    print(f"changed {sorted(changed)}:")
    for (instruction, _), outcome in zip(STEPS, build(changed)):
        print(f"  {outcome:>7}  {instruction}")

In CI, runners start with an empty cache. Reuse it with BuildKit cache exports (`docker buildx build --cache-from type=registry,ref=... --cache-to type=registry,ref=...,mode=max` or `type=gha`), and use cache mounts so package managers keep their download cache between builds without baking it into a layer: `RUN --mount=type=cache,target=/root/.cache/pip pip install -r requirements.txt`.

## Dockerfile instructions

| Instruction | Purpose and gotchas |
|---|---|
| `FROM image:tag AS name` | Base image; pin a version (`python:3.12.7-slim-bookworm`) or a digest. `AS` names a stage. |
| `RUN` | Executes at build time and creates a layer. Chain install and cleanup in one `RUN`; files deleted in a later layer still ship. |
| `COPY` / `ADD` | Copy from the build context or another stage (`--from=builder`). Prefer `COPY`; `ADD` also fetches URLs and unpacks archives, which is rarely wanted. Use `--chown`. |
| `WORKDIR` | Sets and creates the working directory. |
| `ENV` / `ARG` | `ENV` persists into the running container; `ARG` exists only during the build. Neither is safe for secrets: both are visible in image history. |
| `USER` | Run as a non-root user. |
| `EXPOSE` | Documentation only; publishing a port is `-p` or Compose `ports:`. |
| `ENTRYPOINT` / `CMD` | The executable and its default arguments. |
| `HEALTHCHECK` | Command whose exit code marks the container healthy. Kubernetes ignores it and uses probes instead. |

### CMD versus ENTRYPOINT

`ENTRYPOINT` fixes the executable; `CMD` supplies default arguments that `docker run image <args>` replaces. Use the **exec form** (JSON array). The shell form (`CMD python job.py`) runs under `/bin/sh -c`, so the shell is PID 1 and `SIGTERM` from `docker stop` or a Kubernetes eviction never reaches Python; the job is killed hard after the grace period, losing in-flight work and checkpoints. If the process spawns children, add a tiny init (`docker run --init` or `tini`) to reap zombies and forward signals.

```dockerfile
ENTRYPOINT ["python", "-m", "etl.cli"]
CMD ["--date", "today"]              # docker run etl-image --date 2026-11-01 overrides only the args
```

## A production Dockerfile for a Python data job

Multi-stage builds compile wheels in a builder stage that has compilers, then copy only the installed packages into a slim runtime stage. The result is smaller, has fewer CVEs, and contains no build tools.

```dockerfile
# syntax=docker/dockerfile:1.7
FROM python:3.12-slim-bookworm AS builder
ENV PIP_DISABLE_PIP_VERSION_CHECK=1
RUN apt-get update && apt-get install -y --no-install-recommends build-essential \
    && rm -rf /var/lib/apt/lists/*
WORKDIR /build
COPY requirements.lock .
RUN --mount=type=cache,target=/root/.cache/pip \
    pip install --prefix=/install -r requirements.lock

FROM python:3.12-slim-bookworm AS runtime
ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1
RUN useradd --create-home --uid 10001 etl
COPY --from=builder /install /usr/local
WORKDIR /app
COPY --chown=etl:etl src/ ./src/
USER 10001
ENTRYPOINT ["python", "-m", "src.etl.main"]
```

With `uv`, the builder stage becomes `COPY pyproject.toml uv.lock ./` then `RUN uv sync --frozen --no-dev --no-install-project`, and the runtime copies the `.venv`.

Image size checklist: slim or distroless base (Alpine's musl libc breaks many manylinux wheels and can make pandas or pyarrow builds slow), `--no-install-recommends` and apt list cleanup in the same `RUN`, no pip cache, multi-stage builds, a tight `.dockerignore`, and no test data or notebooks in the image. Inspect what each layer adds with `docker history` or `dive`.

The next cell lints a Dockerfile for the most common review comments.

### Where you'll use it

- **Pinning dependencies:** a `Dockerfile` that installs an exact Python and driver version is how a data team keeps last month's job running unchanged.

In [ ]:
import re

DOCKERFILE = """\
FROM python:latest
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
ENV DB_PASSWORD=hunter2
CMD python job.py
"""


def lint(dockerfile: str) -> list[str]:
    lines = [line.strip() for line in dockerfile.splitlines() if line.strip() and not line.startswith("#")]
    findings = []
    for line in lines:
        if re.match(r"FROM\s+\S+(:latest)?\s*$", line) and (":" not in line.split()[1] or line.endswith(":latest")):
            findings.append("unpinned base image")
        if re.match(r"(ENV|ARG)\s+\S*(PASSWORD|SECRET|TOKEN|KEY)", line, re.I):
            findings.append("secret baked into image metadata")
        if re.match(r"(CMD|ENTRYPOINT)\s+[^\[]", line):
            findings.append("shell-form entrypoint: signals not forwarded")
    copy_all = next((i for i, line in enumerate(lines) if re.match(r"COPY\s+\.\s", line)), None)
    install = next((i for i, line in enumerate(lines) if "pip install" in line), None)
    if copy_all is not None and install is not None and copy_all < install:
        findings.append("COPY . before dependency install: cache busts on every edit")
    if not any(line.startswith("USER") for line in lines):
        findings.append("runs as root")
    return findings


for finding in lint(DOCKERFILE):
    print("-", finding)
assert len(lint(DOCKERFILE)) == 5

GOOD = """\
FROM python:3.12-slim-bookworm
WORKDIR /app
COPY requirements.lock .
RUN pip install --no-cache-dir -r requirements.lock
COPY src/ ./src/
USER 10001
ENTRYPOINT ["python", "-m", "src.etl.main"]
"""
assert lint(GOOD) == []

## The build context and .dockerignore

`docker build .` sends the whole directory (the *context*) to the builder. Without a `.dockerignore`, that includes `.git`, virtualenvs, local data extracts and `.env` files: builds get slow, `COPY . .` busts the cache on unrelated changes, and secrets end up in layers. Typical entries:

```text
.git
.venv
__pycache__/
*.pyc
data/
*.parquet
notebooks/
.env
**/.pytest_cache
```

In [ ]:
import fnmatch
from pathlib import PurePosixPath

IGNORE = [".git", ".venv", "__pycache__", "*.pyc", "data", "*.parquet", ".env"]
FILES = {
    "src/etl/main.py": 4_000,
    "src/etl/__pycache__/main.cpython-312.pyc": 9_000,
    "requirements.lock": 2_000,
    ".env": 200,
    ".git/objects/pack/pack-1.pack": 180_000_000,
    ".venv/lib/python3.12/site-packages/pandas/core.so": 60_000_000,
    "data/raw/orders_2026.parquet": 900_000_000,
}


def ignored(path: str) -> bool:
    parts = PurePosixPath(path).parts
    return any(fnmatch.fnmatch(part, pattern) for part in parts for pattern in IGNORE)


context = {path: size for path, size in FILES.items() if not ignored(path)}
print("sent to builder:", sorted(context))
print(f"context size: {sum(FILES.values()) / 1e6:,.0f} MB -> {sum(context.values()) / 1e3:.0f} KB")
assert sorted(context) == ["requirements.lock", "src/etl/main.py"]

## Configuration and secrets

+ `ARG` values are build-time only; `ENV` values persist at runtime. Both show up in `docker history`, so never pass secrets through either.
+ **Build-time secrets** (a private package index token): BuildKit secret mounts, `RUN --mount=type=secret,id=pip_token ...`, which never land in a layer.
+ **Runtime secrets**: inject at run time as files (Docker/Compose secrets mounted at `/run/secrets/<name>`, Kubernetes Secrets, a secrets manager CSI driver) or environment variables from the orchestrator. Files are preferred because environment variables leak into process listings, crash dumps and child processes.

A small helper that reads a mounted secret and falls back to an environment variable for local development:

In [ ]:
import os
import tempfile
from pathlib import Path


def read_secret(name: str, secrets_dir: str = "/run/secrets", default: str | None = None) -> str | None:
    """Read a secret from a mounted file, falling back to an env var for local dev."""
    path = Path(secrets_dir) / name
    if path.is_file():
        return path.read_text(encoding="utf-8").strip()
    return os.environ.get(name.upper(), default)


with tempfile.TemporaryDirectory() as secrets_dir:
    (Path(secrets_dir) / "warehouse_password").write_text("from-file\n", encoding="utf-8")
    os.environ["API_TOKEN"] = "from-env"
    assert read_secret("warehouse_password", secrets_dir) == "from-file"
    assert read_secret("api_token", secrets_dir) == "from-env"
    assert read_secret("missing", secrets_dir, default="unset") == "unset"
print("secret lookup order: mounted file, then environment variable, then default")

## Storage: volumes and bind mounts

| Type | What it is | Use for |
|---|---|---|
| Named volume | Storage managed by Docker, survives container removal | Postgres data, Airflow metadata DB, MinIO data in a local stack |
| Bind mount | A host path mounted into the container | Live-editing DAGs or dbt models in development (often read-only) |
| tmpfs | Memory-backed, discarded on stop | Scratch space for secrets or temporary files |

`docker compose down` keeps named volumes; `down -v` removes them. Bind mounts on macOS and Windows go through a virtual machine file-sharing layer, so heavy I/O (Spark shuffle on a bind mount) is slow; use a volume. File ownership is a frequent problem: a container running as UID 10001 cannot write to a host directory owned by your user unless permissions match.

## Networking

Containers on the same user-defined network (Compose creates one per project) resolve each other by service name through Docker's embedded DNS: Airflow connects to `postgres:5432`, not `localhost`. Inside a container, `localhost` is the container itself. `ports: ["8080:8080"]` publishes a port to the host; services that only talk to each other need no published port. `host.docker.internal` reaches the host from Docker Desktop.

## Docker Compose for a local data stack

Compose describes multi-container stacks in YAML. A local Airflow + Postgres + MinIO + Trino stack:

```yaml
services:
  postgres:
    image: postgres:16
    environment:
      POSTGRES_USER: airflow
      POSTGRES_PASSWORD_FILE: /run/secrets/pg_password
      POSTGRES_DB: airflow
    secrets: [pg_password]
    volumes: [pgdata:/var/lib/postgresql/data]
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U airflow"]
      interval: 5s
      retries: 10

  airflow-init:
    image: my-airflow:2.10.3
    command: ["bash", "-c", "airflow db migrate && airflow users create --role Admin --username admin --password admin --email a@b.c --firstname a --lastname b"]
    depends_on:
      postgres: { condition: service_healthy }

  airflow-scheduler:
    image: my-airflow:2.10.3
    command: ["airflow", "scheduler"]
    volumes: ["./dags:/opt/airflow/dags:ro"]
    depends_on:
      airflow-init: { condition: service_completed_successfully }
    deploy:
      resources:
        limits: { cpus: "2", memory: 4g }

  airflow-webserver:
    image: my-airflow:2.10.3
    command: ["airflow", "webserver"]
    ports: ["8080:8080"]
    depends_on:
      airflow-init: { condition: service_completed_successfully }

  minio:
    image: minio/minio:RELEASE.2024-10-13T13-34-11Z
    command: ["server", "/data", "--console-address", ":9001"]
    volumes: [miniodata:/data]
    healthcheck:
      test: ["CMD", "mc", "ready", "local"]
      interval: 5s

  trino:
    image: trinodb/trino:463
    ports: ["8081:8080"]
    volumes: ["./trino/catalog:/etc/trino/catalog:ro"]
    depends_on:
      minio: { condition: service_healthy }

volumes: {pgdata: {}, miniodata: {}}
secrets:
  pg_password: { file: ./secrets/pg_password.txt }
```

Plain `depends_on: [postgres]` only waits for the container to *start*, not for Postgres to accept connections. `condition: service_healthy` waits for the healthcheck, and `service_completed_successfully` waits for one-off jobs like migrations. Applications should still retry connections: a healthcheck narrows the race, it does not remove it. The next cell derives the startup waves Compose would follow from that file.

### Where you'll use it

- **Local development:** Kafka, Postgres and a Spark master started together on a laptop, so a pipeline can be tested end to end before it reaches the shared cluster.

In [ ]:
from graphlib import TopologicalSorter

import yaml

COMPOSE = yaml.safe_load("""
services:
  postgres: {healthcheck: {test: ["CMD-SHELL", "pg_isready"]}}
  airflow-init: {depends_on: {postgres: {condition: service_healthy}}}
  airflow-scheduler: {depends_on: {airflow-init: {condition: service_completed_successfully}}}
  airflow-webserver: {depends_on: {airflow-init: {condition: service_completed_successfully}}}
  minio: {healthcheck: {test: ["CMD", "mc", "ready", "local"]}}
  trino: {depends_on: {minio: {condition: service_healthy}}}
""")

services = COMPOSE["services"]
for name, service in services.items():
    for dependency, rule in service.get("depends_on", {}).items():
        if rule["condition"] == "service_healthy":
            assert "healthcheck" in services[dependency], f"{name} waits on {dependency}, which has no healthcheck"

sorter = TopologicalSorter({name: set(s.get("depends_on", {})) for name, s in services.items()})
sorter.prepare()
waves = []
while sorter.is_active():
    ready = sorted(sorter.get_ready())
    waves.append(ready)
    sorter.done(*ready)
for number, wave in enumerate(waves, 1):
    print(f"wave {number}: {wave}")
assert waves[0] == ["minio", "postgres"] and waves[-1] == ["airflow-scheduler", "airflow-webserver"]

## Resource limits

Without limits a container can use all host CPU and memory. `--memory 4g` (Compose `deploy.resources.limits.memory`) sets a cgroup hard limit: exceeding it gets the process **OOM-killed** (exit code 137, `OOMKilled: true` in `docker inspect`). `--cpus 2` throttles CPU time rather than killing. Kubernetes requests and limits map onto the same cgroup mechanisms.

For JVM workloads (Spark, Trino, Kafka Connect) size the container from the JVM: the container limit must cover heap **plus** off-heap memory (metaspace, thread stacks, direct buffers, Python worker processes for PySpark). Spark's default overhead is `max(384 MiB, 10% of executor memory)`; PySpark and Arrow-heavy jobs often need more. Modern JVMs respect cgroup limits (`-XX:MaxRAMPercentage=75`) instead of seeing the whole host.

For Airflow, limit worker concurrency to what the container memory supports: eight concurrent pandas tasks at 1.5 GB each will not fit in a 4 GB worker.

In [ ]:
import math

OVERHEAD_FLOOR_MIB = 384
OVERHEAD_FRACTION = 0.10


def executor_container_mib(heap_gib: float, pyspark_memory_gib: float = 0.0) -> int:
    heap_mib = heap_gib * 1024
    overhead = max(OVERHEAD_FLOOR_MIB, heap_mib * OVERHEAD_FRACTION)
    return math.ceil(heap_mib + overhead + pyspark_memory_gib * 1024)


jvm_only = executor_container_mib(8)
with_python = executor_container_mib(8, pyspark_memory_gib=2)
small = executor_container_mib(1)
print(f"8g heap -> {jvm_only} MiB; with 2g Python workers -> {with_python} MiB; 1g heap -> {small} MiB")
assert jvm_only == 9012 and with_python == 11060 and small == 1408


def tasks_that_fit(container_gib: float, per_task_gib: float, reserved_gib: float = 0.5) -> int:
    return int((container_gib - reserved_gib) // per_task_gib)


assert tasks_that_fit(4, 1.5) == 2  # not 8: set worker concurrency accordingly

## Security

+ Run as a **non-root user** (`USER 10001`); in Kubernetes add `runAsNonRoot: true` and a read-only root filesystem where possible.
+ Use minimal, pinned base images and rebuild regularly to pick up OS patches.
+ **Scan** images in CI (Trivy, Grype, Docker Scout, registry scanning) and fail on critical, fixable CVEs: `trivy image --exit-code 1 --severity CRITICAL --ignore-unfixed my-etl:abc123`.
+ Generate an SBOM (`syft`, `docker buildx build --sbom=true`) and sign images (cosign) so clusters can verify provenance.
+ Never mount the Docker socket into a container unless you accept that it grants root on the host.

## Tagging, registries and CI/CD

A tag is mutable: `latest` today is different bytes tomorrow. Tag images with the git SHA (plus a human version such as `1.4.2`), and deploy by SHA or digest so every running container maps to one commit and rollback is redeploying the previous tag. A typical pipeline:

```bash
IMAGE=ghcr.io/acme/etl
SHA=$(git rev-parse --short HEAD)
docker buildx build --platform linux/amd64 \
  --cache-from type=registry,ref=$IMAGE:buildcache \
  --cache-to type=registry,ref=$IMAGE:buildcache,mode=max \
  -t $IMAGE:$SHA --push .
trivy image --exit-code 1 --severity CRITICAL --ignore-unfixed $IMAGE:$SHA
docker run --rm $IMAGE:$SHA pytest -q          # test the exact artifact you ship
```

Build for the platform you deploy to: an image built on an Apple Silicon laptop is `arm64` by default and fails with `exec format error` on `amd64` nodes.

### Custom Airflow images

Extend the official image instead of installing packages at container start (`_PIP_ADDITIONAL_REQUIREMENTS` is for quick trials only: it slows every start and is unreproducible). Pin with Airflow's constraints file:

```dockerfile
FROM apache/airflow:2.10.3-python3.12
COPY requirements.txt /requirements.txt
RUN pip install --no-cache-dir -r /requirements.txt \
    --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-2.10.3/constraints-3.12.txt"
COPY --chown=airflow:root dags/ /opt/airflow/dags/
```

## Debugging a failing container

| Symptom | Check |
|---|---|
| Exits immediately | `docker logs <c>`; `docker inspect <c> --format '{{.State.ExitCode}}'`; run the image with `--entrypoint sh` to look around |
| Exit code 137 | OOM kill (check `OOMKilled`) or `SIGKILL` after the stop grace period |
| Exit code 143 | Graceful `SIGTERM` |
| `exec format error` | Wrong CPU architecture |
| Cannot reach another service | Using `localhost` instead of the service name; services on different networks |
| Permission denied on a mount | UID mismatch between container user and host files |
| Works locally, fails in CI | Unpinned base or dependencies; missing build context files excluded by `.dockerignore` |

`docker exec -it <c> sh` opens a shell in a running container; `docker stats` shows live CPU and memory. Write logs to stdout and stderr (with `PYTHONUNBUFFERED=1`) so `docker logs` and Kubernetes log collection see them.

## Docker and Kubernetes

Kubernetes runs OCI images through a container runtime (containerd or CRI-O; it no longer uses the Docker daemon, but Docker-built images run unchanged). What carries over: the image, its entrypoint, its user, and its stdout logging. What changes: Compose `depends_on` becomes init containers and readiness probes, `HEALTHCHECK` becomes liveness and readiness probes, Compose limits become requests and limits, and secrets come from Kubernetes Secrets. See [Kubernetes](../kubernetes/kubernetes.ipynb).

## Common pitfalls

+ `COPY . .` before installing dependencies, so every edit reinstalls everything.
+ `latest` tags and unpinned dependencies, so builds are not reproducible.
+ Secrets in `ENV`, `ARG` or a copied `.env` file.
+ Shell-form entrypoints that swallow `SIGTERM`.
+ No memory limit, or a limit equal to the JVM heap, so executors are OOM-killed.
+ Writing outputs to the container filesystem instead of a volume or object storage.
+ Huge build contexts because `.dockerignore` is missing.

Questions to check yourself:

1. Why does a `requirements.txt` change rebuild everything after it while a source change does not?
2. What is the difference between `depends_on` with and without `condition: service_healthy`?
3. Why must a Spark executor container limit be larger than `spark.executor.memory`?
4. Why deploy by git SHA or digest rather than by `latest`?

### Where you'll use it

- **Release promotion:** CI builds the image once, tags it with the commit, and the same image moves from staging to production.

### Where you'll use it

- **Failed scheduled jobs:** reading the container logs and exit code of a failed Airflow or Kubernetes job is the first step of the on-call runbook.